# City profile v4 — other permanent income and FCM position

v3 measured a comuna's own money as IPP against the Fondo Común Municipal (FCM). v4 applies the SSG
review of that measure (MEED consolidated methodology, annex section 5.3):

- **Own money now also counts other permanent income (OI).** Recurring income that is neither IPP nor
  FCM, such as regular SUBDERE transfers or sales of services. Education and health transfers stay out.
- **Each comuna is labelled as a net contributor to, or net recipient of, the FCM.**

IPP and FCM are still read from the public SIM/BEP release, so the v3 → v4 difference is the OI
top-up alone. OI comes from SSG's line-item sheet, classified by `data/income_classification.csv`.
Capacity is unchanged from v3.

- Capacity: `reviews/oef/cl-municipal-capacity-tier/releases/v1`
- IPP, FCM and FCM position: `reviews/cl-subdere/cl-subdere-sim-bep/releases/2025`
- OI line items: `reviews/cl-ssg/cl-ssg-municipal-income/releases/2025`
- Release contract: `review.md`

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "dataset-review").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "dataset-review").exists(), "run from inside the CityCatalyst-global-data repository"

REVIEWS = ROOT / "dataset-review" / "reviews"
HERE = REVIEWS / "oef" / "cl-city-action-fundability" / "releases" / "v4"

tier = pd.read_csv(
    REVIEWS / "oef/cl-municipal-capacity-tier/releases/v1/data/municipal_capacity_tier.csv",
    dtype={"comuna_cut": str})
sim = pd.read_csv(
    REVIEWS / "cl-subdere/cl-subdere-sim-bep/releases/2025/data/sim_municipal_income_2025.csv",
    dtype={"comuna_cut": str, "source_vintage": str})
lines = pd.read_csv(
    REVIEWS / "cl-ssg/cl-ssg-municipal-income/releases/2025/data/bep_income_lines_2025.csv",
    dtype={"comuna_cut": str})
classes = pd.read_csv(HERE / "data/income_classification.csv", dtype=str)
v3 = pd.read_csv(
    REVIEWS / "oef/cl-city-action-fundability/releases/v3/data/city_finance_profile.csv",
    dtype={"comuna_cut": str})

for frame in (tier, sim, lines, v3):
    frame["comuna_cut"] = frame["comuna_cut"].str.zfill(5)
print(f"capacity {len(tier)} | SIM/BEP {len(sim)} | SSG lines {len(lines)} | classified codes {len(classes)}")

capacity 345 | SIM/BEP 345 | SSG lines 345 | classified codes 65


## Validate the classification

Every classified code must exist in the line-item sheet, and no code may sit inside another one, so
nothing is counted twice. The four classes must also add back to each comuna's total income: every
peso is either IPP, OI, FCM or explicitly excluded with a reason.

In [2]:
CLASSES = {"ipp", "oi_permanent", "fcm", "excluded"}
assert set(classes["income_class"]) == CLASSES
assert classes["budget_code"].is_unique
assert classes["reason"].str.strip().ne("").all()
missing = set(classes["budget_code"]) - set(lines.columns)
assert not missing, missing

def significant(code):
    parts = code.split(".")
    while parts and int(parts[-1]) == 0:
        parts.pop()
    return parts

sig = {c: significant(c) for c in classes["budget_code"]}
nested = [(a, b) for a in sig for b in sig
          if a != b and sig[b][:len(sig[a])] == sig[a]]
assert not nested, nested

reported = lines[lines["reported"]].set_index("comuna_cut")
subtitles = [c for c in lines.columns if c.startswith("115.") and c.split(".")[2] == "00"]
total = reported[subtitles].sum(axis=1)
by_class = {k: reported[classes.loc[classes["income_class"].eq(k), "budget_code"]].sum(axis=1)
            for k in CLASSES}
assert (sum(by_class.values()) - total).eq(0).all(), "classes do not add up to total income"
print(f"classification is a complete partition of total income for {len(reported)} reporting comunas")

classification is a complete partition of total income for 340 reporting comunas


## Check the IPP and FCM codes against SIM/BEP

The IPP and FCM rows in the classification exist to keep OI from overlapping them. They should
reproduce SIM/BEP's published IPP and FCM wherever the two snapshots agree. SSG's sheet is a
different download from the SIM/BEP release, so some comunas disagree on totals; those are reported,
not corrected.

In [3]:
s = sim.set_index("comuna_cut").loc[reported.index]
same_snapshot = ((by_class["fcm"] - s["fcm_percibido_mclp"]).abs() <= 2) & \
                ((total - s["ingresos_totales_percibido_mclp"]).abs() <= 5)
ipp_exact = ((by_class["ipp"] - s["ipp_percibido_mclp"]).abs() <= 3) & same_snapshot

print(f"snapshots agree on FCM and total income: {same_snapshot.sum()} of {len(s)} reporting comunas")
print(f"IPP codes reproduce SIM/BEP IPP exactly: {ipp_exact.sum()} of {same_snapshot.sum()}")
assert ipp_exact.sum() >= 0.95 * same_snapshot.sum(), "IPP code set no longer matches SUBDERE's definition"

blank = lines.loc[~lines["reported"], ["comuna_cut", "comuna"]]
print("comunas with no line items (keep their v3 autonomy):", blank["comuna"].tolist())

snapshots agree on FCM and total income: 298 of 340 reporting comunas
IPP codes reproduce SIM/BEP IPP exactly: 286 of 298
comunas with no line items (keep their v3 autonomy): ['SAN PEDRO DE ATACAMA', 'RINCONADA', 'LOTA', 'CURACAUTÍN', 'FRUTILLAR']


## Recalculate autonomy

`autonomy = (IPP + OI) / (IPP + OI + FCM)`, with IPP and FCM from SIM/BEP and OI from the classified
line items, on the cash-received basis. Where a comuna has no line items its OI is unknown, so it
keeps the v3 value and `oi_reported` is false.

FCM position reads SIM/BEP's ratio of what a comuna pays into the FCM to what it receives. Above 1 it
is a net contributor.

In [4]:
prof = tier.merge(
    sim[["comuna_cut", "ipp_percibido_mclp", "fcm_percibido_mclp", "fcm_contribution_ratio", "source_vintage"]],
    on="comuna_cut", how="inner", validate="one_to_one")
oi = by_class["oi_permanent"].rename("oi_permanent_mclp")
prof = prof.merge(oi, left_on="comuna_cut", right_index=True, how="left", validate="one_to_one")
prof["oi_reported"] = prof["oi_permanent_mclp"].notna()
prof["oi_permanent_mclp"] = prof["oi_permanent_mclp"].fillna(0).astype(int)
prof = prof.rename(columns={"ipp_percibido_mclp": "ipp_mclp", "fcm_percibido_mclp": "fcm_mclp"})

own = prof["ipp_mclp"] + prof["oi_permanent_mclp"]
prof["autonomy"] = (own / (own + prof["fcm_mclp"])).clip(0, 1).round(6)

assert prof["fcm_contribution_ratio"].notna().all()
prof["fcm_position"] = prof["fcm_contribution_ratio"].gt(1).map(
    {True: "net_contributor", False: "net_recipient"})

ARCHETYPE = {
    (True, True): "Self-sufficient",
    (False, True): "Delivery-ready",
    (True, False): "Well-resourced",
    (False, False): "Support-ready",
}
prof["city_archetype"] = [ARCHETYPE[(a >= 0.5, c >= 0.5)]
                          for a, c in zip(prof["autonomy"], prof["capacity"])]
prof["autonomy_basis"] = "sim_bep_ipp_fcm_plus_ssg_oi_percibido"
prof = prof.rename(columns={"source_vintage": "autonomy_source_vintage"})
print(prof["fcm_position"].value_counts().to_string())

fcm_position
net_recipient      328
net_contributor     17


## Validate

OI can only raise autonomy, never lower it. Comunas without line items must equal v3. Every net
contributor already clears the 0.5 line, so FCM position is carried as a label and does not need a
rule of its own; the last assertion fails if that ever stops being true.

In [5]:
assert len(prof) == 345 and prof["comuna_cut"].is_unique
assert prof[["capacity", "autonomy", "city_archetype", "fcm_position"]].notna().all().all()
assert prof["autonomy"].between(0, 1).all() and prof["capacity"].between(0, 1).all()
assert (prof["oi_permanent_mclp"] >= 0).all()

cmp = prof.merge(v3[["comuna_cut", "autonomy", "city_archetype", "capacity"]],
                 on="comuna_cut", suffixes=("", "_v3"), validate="one_to_one")
assert cmp["capacity"].equals(cmp["capacity_v3"]), "capacity changed"
assert (cmp["autonomy"] >= cmp["autonomy_v3"] - 1e-6).all(), "OI lowered autonomy"
no_oi = ~cmp["oi_reported"]
assert (cmp.loc[no_oi, "autonomy"] - cmp.loc[no_oi, "autonomy_v3"]).abs().max() <= 1e-6

contributors = prof[prof["fcm_position"].eq("net_contributor")]
assert (contributors["autonomy"] >= 0.5).all(), "a net contributor fell below 0.5: FCM position now needs a rule"
print(f"{len(contributors)} net contributors, lowest autonomy {contributors['autonomy'].min():.3f}")

17 net contributors, lowest autonomy 0.554


## What changed from v3

In [6]:
cmp["autonomy_change"] = (cmp["autonomy"] - cmp["autonomy_v3"]).round(6)
print(cmp["autonomy_change"].describe().round(4).to_string())
moved = cmp[cmp["city_archetype"] != cmp["city_archetype_v3"]]
print(f"\n{len(moved)} comunas change archetype:")
print(moved[["comuna", "autonomy_v3", "autonomy", "city_archetype_v3", "city_archetype"]]
      .sort_values("comuna").to_string(index=False))
print()
print(prof["city_archetype"].value_counts().sort_index().to_string())

count    345.0000
mean       0.0147
std        0.0105
min        0.0000
25%        0.0080
50%        0.0131
75%        0.0177
max        0.0765

1 comunas change archetype:
 comuna  autonomy_v3  autonomy city_archetype_v3  city_archetype
Natales     0.499992  0.513426    Delivery-ready Self-sufficient

city_archetype
Delivery-ready     104
Self-sufficient     66
Support-ready      150
Well-resourced      25


## Export

Same columns as v3, plus the three money components, the OI flag and FCM position. The modelled
table keeps selecting only its established columns, so no schema migration is implied.

In [7]:
OUT_COLS = [
    "comuna_cut", "comuna", "population", "tramo", "tramo_label",
    "gl_technical_capacity", "capacity",
    "ipp_mclp", "oi_permanent_mclp", "fcm_mclp", "autonomy", "oi_reported",
    "fcm_contribution_ratio", "fcm_position", "city_archetype",
    "capacity_basis", "autonomy_basis", "autonomy_source_vintage", "t4_provisional",
]
out = prof[OUT_COLS].sort_values("comuna_cut").reset_index(drop=True)
out.to_csv(HERE / "data/city_finance_profile.csv", index=False)
print(f"wrote data/city_finance_profile.csv — {out.shape[0]} rows x {out.shape[1]} columns")

wrote data/city_finance_profile.csv — 345 rows x 19 columns


## Result

345 complete profiles. Autonomy now includes other permanent income for the 340 comunas with line
items. The five without line items keep their v3 value. FCM position is recorded for all 345 and
changes no route. The v3 open decisions (T4, T1 gate, Padre Hurtado, Natales) still apply, and the
OI input adds its own licence question. Both are in `review.md`.